# Week 13 — Dashboards, Streaming, and Deployment

**IT 2012 · Unstructured Data**  

This notebook contains live demos for the Week 13 lecture. Run the cells top to bottom.
Each Dash app runs inline in the notebook on its own port, restart the kernel between
sections if you hit port-in-use errors.

**Sections**
1. From static to interactive (a Week 12 recap)
2. Your first Dash application
3. Layout components: `html` and `dcc`
4. Your first callback
5. Multi-input and multi-output callbacks
6. Styling with inline, `className`, and external CSS
7. Live data with `dcc.Interval`
8. Memory-bounded streaming patterns
9. Dockerfile, `requirements.txt`, and `compose.yaml`
10. Hands-on exercises


## Setup

Install the packages once. The `# !` prefix runs the command in the shell from the notebook —
remove the `#` for the first run, then put it back.


In [6]:
!pip install dash pandas plotly
%pip install dash


  Using cached dash-4.1.0-py3-none-any.whl.metadata (11 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached werkzeug-3.1.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached retrying-1.4.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached narwhals-2.21.2-py3-none-any.whl.metadata (16 kB)
Using cached dash-4.1.0-py3-none-any.whl (7.2 MB)
Using cached flask-3.1.3-py3-none-any.whl (103 kB)
Using cached werkzeug-3.1.8-py3-none-any.whl (226 kB)
Using cached plotly-6.7.0-py3-none-any.whl (9.9 MB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached itsdangerous-2.2.0-py3-none-any.whl (16 kB)
Using cached narwhals-2.21.2-py3-none-any.whl (451 kB)
Using cached retrying-1.4.2-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [dash]2m 9/1

In [7]:
import os
import time
import random
from collections import deque
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from dash import Dash, html, dcc, callback, Output, Input, State

# Each demo uses a different port so they can co-exist
PORTS = {
    'minimal': 8051,
    'components': 8052,
    'first_callback': 8053,
    'multi_io': 8054,
    'styling': 8055,
    'live': 8056,
}
print('Dash version:', __import__('dash').__version__)


Dash version: 4.1.0


---
## 1. From static to interactive

Last week we built static charts with Plotly. They are great for reports, but the reader
sees one fixed view. If they want to filter by source or narrow to a date range, they have
to ask us to re-run the notebook. Let's reproduce that pattern first so we can compare.

In [ ]:
# Load the health & wellness articles dataset
csv_path = os.path.join('..', 'data', 'processed', 'cleaned', 'articles_clean.csv')
df = pd.read_csv(csv_path)
df['published_year'] = pd.to_numeric(df['published_year'], errors='coerce')
df['publishedAt'] = pd.to_datetime(df['publishedAt'], errors='coerce', utc=True)
df['published_month'] = df['publishedAt'].dt.to_period('M').dt.to_timestamp()

# Top sources and categories used across all demo filters
sources = df['source_name'].value_counts().nlargest(6).index.tolist()
categories = sorted(df['Category'].dropna().unique().tolist())

print(f"Articles loaded: {len(df):,}")
df[['source_name', 'title', 'Category', 'Word Count', 'published_year']].head()

In [ ]:
# A static chart — one fixed view. The reader cannot filter.
top_df = df[df['source_name'].isin(sources)].copy()
monthly = top_df.groupby(['published_month', 'source_name']).size().reset_index(name='articles')
fig = px.line(monthly, x='published_month', y='articles', color='source_name',
              title='Monthly articles by source')
fig.show()

---
## 2. Your first Dash application

A Dash app has four parts:
1. **Create the app**: `app = Dash(__name__)`
2. **Define the layout**: a tree of `html` and `dcc` components
3. **Register callbacks**: Python functions decorated with `@callback`
4. **Run the server**: `app.run(...)`

Below is the smallest useful app — one title and one chart. No callbacks yet.

> Running in the notebook: `jupyter_mode='inline'` shows the app embedded in the cell output.
> If the cell looks stuck, look for the Dash output below — it has rendered.


In [ ]:
app1 = Dash(__name__)

monthly_all = df.groupby('published_month').size().reset_index(name='articles')

app1.layout = html.Div([
    html.H1('Health & Wellness Information Dashboard'),
    html.P('Monthly article ingestion from the Health & Wellness pipeline.'),
    dcc.Graph(figure=px.line(monthly_all, x='published_month', y='articles',
                             title='Articles ingested per month')),
])

app1.run(jupyter_mode='inline', port=PORTS['minimal'], debug=False)

---
## 3. Layout components: `html` and `dcc`

The layout is a tree. The `dash.html` module mirrors HTML tags one-to-one. The `dash.dcc`
module provides interactive components: dropdowns, sliders, date pickers, graphs.

The app below shows several common components. There is no callback yet, so the controls
do not actually change anything — we are just rendering the layout.


In [ ]:
app2 = Dash(__name__)

app2.layout = html.Div([
    html.H2('Component catalog'),

    html.Label('Category (dropdown)'),
    dcc.Dropdown(
        id='category-dd',
        options=[{'label': c, 'value': c} for c in categories],
        value=categories[0] if categories else None,
    ),

    html.Br(),
    html.Label('Year range (slider)'),
    dcc.RangeSlider(
        id='year-range',
        min=2020, max=2026, step=1, value=[2020, 2026],
        marks={y: str(y) for y in range(2020, 2027)},
    ),

    html.Br(), html.Br(),
    html.Label('Search article title'),
    dcc.Input(
        id='search',
        type='text',
        placeholder='Type a keyword...',
        debounce=True,
        style={'width': '100%', 'padding': '6px'},
    ),

    html.Br(), html.Br(),
    html.Label('Sources (checklist)'),
    dcc.Checklist(
        id='sources-check',
        options=[{'label': s, 'value': s} for s in sources],
        value=sources[:3],
        inline=True,
    ),

    dcc.Graph(id='chart'),
], style={'maxWidth': '900px', 'margin': '20px auto', 'fontFamily': 'sans-serif'})

app2.run(jupyter_mode='inline', port=PORTS['components'], debug=False)

---
## 4. Your first callback

A callback is a Python function decorated with `@callback`. The decorator declares which
component property is the **Input** and which is the **Output**. When the Input changes,
Dash re-runs the function and writes the return value into the Output.

Here, changing the dropdown updates the chart.


In [ ]:
app3 = Dash(__name__)

app3.layout = html.Div([
    html.H2('Articles by category — pick one'),
    dcc.Dropdown(
        id='category-pick',
        options=[{'label': c, 'value': c} for c in categories],
        value=categories[0] if categories else None,
    ),
    dcc.Graph(id='category-chart'),
], style={'maxWidth': '900px', 'margin': '20px auto', 'fontFamily': 'sans-serif'})


@callback(
    Output('category-chart', 'figure'),
    Input('category-pick', 'value'),
)
def update_chart(category):
    filtered = df[df['Category'] == category] if category else df
    monthly = filtered.groupby('published_month').size().reset_index(name='count')
    return px.bar(monthly, x='published_month', y='count',
                  title=f'Monthly articles — {category}')


app3.run(jupyter_mode='inline', port=PORTS['first_callback'], debug=False)

---
## 5. Multi-input and multi-output callbacks

A callback can have several Inputs (all listed in the decorator) and several Outputs
(returned as a tuple). The app below has:
- Two Inputs: category dropdown and year range slider
- Two Outputs: the chart figure and a small KPI text

Use `State` to read a property without triggering the callback — useful for "click button
to submit" flows where text fields are State and the button click is Input.

In [ ]:
app4 = Dash(__name__)

app4.layout = html.Div([
    html.H2('Article explorer'),

    html.Div([
        html.Label('Category'),
        dcc.Dropdown(id='cat', options=[{'label': c, 'value': c} for c in categories],
                     value=categories[0] if categories else None),
    ], style={'width': '40%', 'display': 'inline-block', 'paddingRight': '20px'}),

    html.Div([
        html.Label('Year range'),
        dcc.RangeSlider(id='yr', min=2020, max=2026, step=1, value=[2020, 2026],
                        marks={y: str(y) for y in range(2020, 2027)}),
    ], style={'width': '55%', 'display': 'inline-block'}),

    html.Hr(),
    html.Div(id='kpi', style={'fontSize': '20px', 'fontWeight': 'bold',
                              'margin': '10px 0', 'color': '#003a6d'}),
    dcc.Graph(id='exp-chart'),
], style={'maxWidth': '900px', 'margin': '20px auto', 'fontFamily': 'sans-serif'})


@callback(
    Output('exp-chart', 'figure'),
    Output('kpi', 'children'),
    Input('cat', 'value'),
    Input('yr', 'value'),
)
def explore(category, year_range):
    filtered = df.copy()
    if category:
        filtered = filtered[filtered['Category'] == category]
    if year_range:
        filtered = filtered[filtered['published_year'].between(year_range[0], year_range[1])]
    monthly = filtered.groupby('published_month').size().reset_index(name='count')
    avg_words = pd.to_numeric(filtered['Word Count'], errors='coerce').mean()
    fig = px.bar(monthly, x='published_month', y='count',
                 title=f'{category or "All categories"} — {year_range[0]}–{year_range[1]}')
    kpi = f'{category or "All"}: {len(filtered):,} articles · avg {avg_words:.0f} words'
    return fig, kpi


app4.run(jupyter_mode='inline', port=PORTS['multi_io'], debug=False)

---
## 6. Styling: inline, `className`, and external CSS

Three options:

1. **Inline `style={...}`** — quickest, scoped to one component. Property names use
   camelCase (`textAlign`, not `text-align`).
2. **`className="..."`** with CSS in a file under `assets/` next to your app — best for
   re-used styles.
3. **dash-bootstrap-components** — pre-built responsive cards, rows, navbars.


In [ ]:
# Inline styling demo — health-themed stat cards

card_style = {
    'padding': '20px',
    'border': '1px solid #cdd7e3',
    'borderRadius': '8px',
    'backgroundColor': '#fafbfd',
    'marginBottom': '10px',
    'fontFamily': 'sans-serif',
}

total_articles = len(df)
unique_sources = df['source_name'].nunique()

app5 = Dash(__name__)

app5.layout = html.Div([
    html.H2('Health & Wellness Pipeline — stats', style={'color': '#003a6d'}),

    html.Div([
        html.H3('Total Articles', style={'color': '#e27d35', 'marginTop': 0}),
        html.P(f'{total_articles:,} articles ingested across all sources.'),
    ], style=card_style),

    html.Div([
        html.H3('Unique Sources', style={'color': '#33a86e', 'marginTop': 0}),
        html.P(f'{unique_sources} distinct news sources in the pipeline.'),
    ], style=card_style),

], style={'maxWidth': '700px', 'margin': '20px auto'})

app5.run(jupyter_mode='inline', port=PORTS['styling'], debug=False)

---
## 7. Live data with `dcc.Interval`

`dcc.Interval` is an invisible component that ticks every `interval` milliseconds. On every
tick its `n_intervals` property increases. Use `n_intervals` as a callback Input, and the
callback re-runs on every tick.

Below we simulate a live article ingestion stream: a deque of the most recent 50 readings,
refreshed every 2 seconds with a simulated articles-per-second rate.

In [ ]:
# Shared in-memory buffer — simulates a live article ingestion source
live_buffer = deque(maxlen=50)
now = datetime.now()
for i in range(20):
    live_buffer.append({
        'time': now - timedelta(seconds=(20 - i) * 2),
        'value': 50 + np.random.normal(0, 5),
    })


app6 = Dash(__name__)

app6.layout = html.Div([
    html.H2('Live article ingestion rate (simulated)'),
    html.P('Refreshes every 2 seconds. The buffer keeps the last 50 readings.'),
    dcc.Graph(id='live-chart'),
    dcc.Interval(id='tick', interval=2000, n_intervals=0),
], style={'maxWidth': '900px', 'margin': '20px auto', 'fontFamily': 'sans-serif'})


@callback(
    Output('live-chart', 'figure'),
    Input('tick', 'n_intervals'),
)
def refresh(n):
    # Append a new simulated ingestion-rate reading every tick
    live_buffer.append({
        'time': datetime.now(),
        'value': 50 + np.random.normal(0, 5),
    })
    data = list(live_buffer)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=[d['time'] for d in data],
        y=[d['value'] for d in data],
        mode='lines+markers',
        line=dict(color='#003a6d', width=2),
    ))
    fig.update_layout(
        title=f'Articles / sec — tick #{n}',
        yaxis_range=[20, 80],
        margin=dict(l=40, r=20, t=50, b=40),
        height=400,
    )
    return fig


app6.run(jupyter_mode='inline', port=PORTS['live'], debug=False)

---
## 8. Memory-bounded streaming patterns

Live data never stops arriving. Four techniques keep memory bounded.

### 8.1 Buffering — `collections.deque(maxlen=N)`

A deque with `maxlen=N` automatically drops the oldest element when a new one is appended
past the limit. Memory stays O(N) regardless of how long the program runs.


In [16]:
buf = deque(maxlen=5)
for i in range(12):
    buf.append(i)
    print(f'after append {i}: {list(buf)}')


after append 0: [0]
after append 1: [0, 1]
after append 2: [0, 1, 2]
after append 3: [0, 1, 2, 3]
after append 4: [0, 1, 2, 3, 4]
after append 5: [1, 2, 3, 4, 5]
after append 6: [2, 3, 4, 5, 6]
after append 7: [3, 4, 5, 6, 7]
after append 8: [4, 5, 6, 7, 8]
after append 9: [5, 6, 7, 8, 9]
after append 10: [6, 7, 8, 9, 10]
after append 11: [7, 8, 9, 10, 11]


### 8.2 Running aggregation

Instead of storing every event, keep just the statistics. A million events collapse to a
few numbers. Welford's algorithm computes mean and variance in one pass without storing
the data.


In [17]:
class RunningStats:
    '''Welford's online algorithm — mean and variance without storing data.'''
    def __init__(self):
        self.n = 0
        self.mean = 0.0
        self._m2 = 0.0
        self.min = float('inf')
        self.max = float('-inf')

    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self._m2 += delta * delta2
        self.min = min(self.min, x)
        self.max = max(self.max, x)

    @property
    def variance(self):
        return self._m2 / self.n if self.n > 1 else 0.0

    @property
    def std(self):
        return self.variance ** 0.5


# Stream a million events through the stats — memory stays constant
stats = RunningStats()
for _ in range(1_000_000):
    stats.update(np.random.normal(100, 15))

print(f'count: {stats.n:,}')
print(f'mean:  {stats.mean:.3f}')
print(f'std:   {stats.std:.3f}')
print(f'min:   {stats.min:.3f}')
print(f'max:   {stats.max:.3f}')


count: 1,000,000
mean:  99.994
std:   15.013
min:   12.269
max:   181.682


### 8.3 Sliding windows

Show only the last N minutes. Combine a deque with a timestamp filter to drop stale points.


In [18]:
def last_5_minutes(events):
    '''Return only events within the last 5 minutes.'''
    cutoff = datetime.now() - timedelta(minutes=5)
    return [e for e in events if e['time'] >= cutoff]


# Build a fake event stream spanning 10 minutes
now = datetime.now()
events = [
    {'time': now - timedelta(minutes=10) + timedelta(seconds=15 * i),
     'value': round(np.random.normal(50, 10), 2)}
    for i in range(40)
]

print(f'Total events:        {len(events)}')
print(f'Last 5 min only:     {len(last_5_minutes(events))}')


Total events:        40
Last 5 min only:     19


### 8.4 Down-sampling

When a chart needs to render too many points, plot every Nth one. The LTTB algorithm
(Largest Triangle Three Buckets) preserves the visual shape with far fewer points than
simple stride sampling.


In [19]:
# Simple stride down-sampling
big = np.random.normal(0, 1, 10_000).cumsum()
print(f'Full series:       {len(big):,} points')

stride = 50
small = big[::stride]
print(f'Stride-sampled:    {len(small)} points (1 in {stride})')

# Plot both for a quick visual comparison
fig = go.Figure()
fig.add_trace(go.Scatter(y=big, mode='lines', name='full', opacity=0.4))
fig.add_trace(go.Scatter(x=np.arange(0, len(big), stride),
                          y=small, mode='lines', name='stride 50'))
fig.update_layout(title='Down-sampling: full vs every 50th point',
                  height=350, margin=dict(l=40, r=20, t=40, b=40))
fig.show()


Full series:       10,000 points
Stride-sampled:    200 points (1 in 50)


---
## 9. Dockerfile, `requirements.txt`, and `compose.yaml`

These cells write the deployment files to a `deploy/` folder next to the notebook. You
cannot run `docker build` inside Jupyter, but you can write the files, open a terminal,
`cd deploy`, and run `docker compose up`.


In [ ]:
deploy_dir = 'deploy'
os.makedirs(deploy_dir, exist_ok=True)


# A minimal health & wellness Dash app we will containerise
APP_PY = '''
from dash import Dash, html, dcc
import plotly.express as px
import pandas as pd
import os

csv_path = os.path.join('data', 'processed', 'cleaned', 'articles_clean.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df['published_year'] = pd.to_numeric(df['published_year'], errors='coerce')
    counts = df.groupby('published_year').size().reset_index(name='articles')
else:
    counts = pd.DataFrame({'published_year': [2024, 2025, 2026], 'articles': [120, 340, 95]})

app = Dash(__name__)
server = app.server   # exposed for gunicorn

app.layout = html.Div([
    html.H1('Health & Wellness Information Dashboard'),
    dcc.Graph(figure=px.bar(counts, x='published_year', y='articles',
                            title='Articles published per year')),
])

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=8050)
'''.strip()

with open(f'{deploy_dir}/app.py', 'w') as f:
    f.write(APP_PY + '\n')


REQUIREMENTS = '''
dash>=2.18
plotly>=5.24
pandas>=2.2
pymongo>=4.0
gunicorn>=23.0
'''.strip()

with open(f'{deploy_dir}/requirements.txt', 'w') as f:
    f.write(REQUIREMENTS + '\n')


DOCKERFILE = '''
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8050
CMD ["gunicorn", "-b", "0.0.0.0:8050", "app:server"]
'''.strip()

with open(f'{deploy_dir}/Dockerfile', 'w') as f:
    f.write(DOCKERFILE + '\n')


COMPOSE = '''
services:
  web:
    build: .
    ports:
      - "8050:8050"
    environment:
      MONGO_URI: mongodb://db:27017
      MONGO_DB: articles_pipeline
      DASH_DEBUG: "false"
  db:
    image: mongo:7
    volumes:
      - mongo-data:/data/db
volumes:
  mongo-data:
'''.strip()

with open(f'{deploy_dir}/compose.yaml', 'w') as f:
    f.write(COMPOSE + '\n')


DOCKERIGNORE = '''
__pycache__
*.pyc
.venv
.git
.ipynb_checkpoints
'''.strip()

with open(f'{deploy_dir}/.dockerignore', 'w') as f:
    f.write(DOCKERIGNORE + '\n')

print('Wrote:')
for f in sorted(os.listdir(deploy_dir)):
    print(f'  {deploy_dir}/{f}')

Open a terminal, then:

```bash
cd deploy
docker build -t health-wellness-dash .
docker run -p 8050:8050 health-wellness-dash
# or, for the full stack with MongoDB:
docker compose up
```

Then open http://localhost:8050 in your browser.

---
## Wrap-up

You now have:
- A minimal Dash app showing Health & Wellness article data (Section 2)
- Layout components rendered without callbacks (Section 3)
- Single-callback (Section 4) and multi-I/O callback (Section 5) apps
- A styled Health & Wellness pipeline stats app (Section 6)
- A live-updating article ingestion rate simulator (Section 7)
- Memory-bounded streaming techniques (Section 8)
- A complete `deploy/` folder ready for `docker compose up` (Section 9)


**Next week:** Pipeline Orchestration and Automation.